# Joining and merging Pandas `DataFrame`s

Joining dataframes involves combining data not based on simple concatenation of rows or columns, but instead based on the values of variables (columns). Specifically, it involves combining data by matching values across datasets. Often you have datasets that share a column but don't share the same kind of rows (observations).

For example, you may have a dataset about hospital visits that contains a Patient ID for each visit. If you have separate dataset with information about each patient that also contains a Patient ID column, you could join them together to create a dataset with patient visits *and* patient information for each visit.

We'll illustrate different types of joins on another example dataset measuring sidewalk quality, where each row is a street segment, a block of a street from one cross street to another.

In [1]:
import pandas as pd

In [2]:
sidewalk_quality = pd.DataFrame(
    {'street': ['Forbes Ave', 'Forbes Ave', 'Oakland Ave', 'Oakland Ave', 'Oakland Ave', 'Sennott St'],
     'beginning_cross_street': ['Atwood St', 'Oakland Ave', 'Fifth Ave', 'Forbes Ave', 'Sennott St', 'Bouquet St'],
     'end_cross_street': ['Oakland Ave', 'Bouquet St', 'Forbes Ave', 'Sennott St', 'Bates St', 'Oakland Ave',],
     'segment_length_m': [80, 85, 100, 100, 550, 110],
     'surface_quality': ['fair', 'good', 'good', 'bad', 'bad', 'fair'],
     'obstructions': [True, True, False, False, False, True]
    })
sidewalk_quality

,street,beginning_cross_street,end_cross_street,segment_length_m,surface_quality,obstructions
0,Forbes Ave,Atwood St,Oakland Ave,80,fair,True
1,Forbes Ave,Oakland Ave,Bouquet St,85,good,True
2,Oakland Ave,Fifth Ave,Forbes Ave,100,good,False
3,Oakland Ave,Forbes Ave,Sennott St,100,bad,False
4,Oakland Ave,Sennott St,Bates St,550,bad,False
5,Sennott St,Bouquet St,Oakland Ave,110,fair,True


Let's say we also have a dataset with information about each street. In this dataset, each row is about the entire street, not just segments of it.

In [3]:
street_info = pd.DataFrame(
    {'street': ['Forbes Ave', 'Oakland Ave', 'Meyran Ave'],
     'street_type': ['arterial', 'residential', 'residential']
    })
street_info

,street,street_type
0,Forbes Ave,arterial
1,Oakland Ave,residential
2,Meyran Ave,residential


## Inner join
Here we will join dataframes on a **key** column (joining variable) that they both have in common. Inner joins keep unique values in the key column that are present in **both** dataframes.

The following diagram illustrates matching on a key column between 2 datasets and merging the result: <img src="attachment:41bd3ea3-8bed-4d2f-8e06-5cff1a7a6b26.png" width="400"/>

*Figure originally from https://r4ds.hadley.nz/joins.html#fig-join-inner*

In [4]:
joined = pd.merge(sidewalk_quality, street_info, on='street')
joined

,street,beginning_cross_street,end_cross_street,segment_length_m,surface_quality,obstructions,street_type
0,Forbes Ave,Atwood St,Oakland Ave,80,fair,True,arterial
1,Forbes Ave,Oakland Ave,Bouquet St,85,good,True,arterial
2,Oakland Ave,Fifth Ave,Forbes Ave,100,good,False,residential
3,Oakland Ave,Forbes Ave,Sennott St,100,bad,False,residential
4,Oakland Ave,Sennott St,Bates St,550,bad,False,residential


That's cool! We now have the correct street type for each of our street segments! But wait. Did we lose some segments?

<img src="attachment:25a6b47e-7f75-431c-bed0-7d49f12dc68e.png" width=400/>

Let's take a look at what was left out of the joined dataset from each.

## Left and right joins
Sometimes you want to keep some of those keys that only occur in one dataframe! Left and right joins allow this kind of join. 

A left join keeps all of the keys that occur in the first dataframe provided to the `pd.merge()` function (the one on the "left") regardless of whether they all have matches in the other (right) dataframe. If keys don't have matches, they are filled with null values (`NaN`) for any columns from the right dataframe. Right joins keep all the keys in the right dataframe.

**A left join is illustrated below.** Note that the output will have a null value for the data from the right dataframe (y) since there are no matches for the key.

<img src="attachment:423442f6-65c6-4ce3-b835-0f6ce4e08eed.png" width=400/>

In [6]:
left_joined = pd.merge(sidewalk_quality, street_info, how='right')
left_joined

,street,beginning_cross_street,end_cross_street,segment_length_m,surface_quality,obstructions,street_type
0,Forbes Ave,Atwood St,Oakland Ave,80.0,fair,True,arterial
1,Forbes Ave,Oakland Ave,Bouquet St,85.0,good,True,arterial
2,Oakland Ave,Fifth Ave,Forbes Ave,100.0,good,False,residential
3,Oakland Ave,Forbes Ave,Sennott St,100.0,bad,False,residential
4,Oakland Ave,Sennott St,Bates St,550.0,bad,False,residential
5,Meyran Ave,NaN,NaN,NaN,NaN,NaN,residential


## Outer (full) joins
Outer joins allow keeping all unique values for the key column present in either dataframe. But watch out for all the `NaN` values!

**Outer join diagram:**

<img src="attachment:f49ca62b-6019-4ac4-a02b-6199c9e0fe17.png" width=400 />

In [8]:
outer_joined = pd.merge(sidewalk_quality, street_info, how='outer')
outer_joined

,street,beginning_cross_street,end_cross_street,segment_length_m,surface_quality,obstructions,street_type
0,Forbes Ave,Atwood St,Oakland Ave,80.0,fair,True,arterial
1,Forbes Ave,Oakland Ave,Bouquet St,85.0,good,True,arterial
2,Meyran Ave,NaN,NaN,NaN,NaN,NaN,residential
3,Oakland Ave,Fifth Ave,Forbes Ave,100.0,good,False,residential
4,Oakland Ave,Forbes Ave,Sennott St,100.0,bad,False,residential
5,Oakland Ave,Sennott St,Bates St,550.0,bad,False,residential
6,Sennott St,Bouquet St,Oakland Ave,110.0,fair,True,NaN
